# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RAJkumar5736602/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Logistic regression (readable), random forest and HistGradientBoosting (non-linear interactions). Pick on held-out P@50 and AUC.

In [1]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p/'data/raw').exists())
sys.path.insert(0, str(root/'work/scripts'))
from wk import *
import pandas as pd, numpy as np
df = load()
print(list(models()))

['logreg', 'random_forest', 'hist_gbm']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**GroupShuffleSplit by client_id, 20% clients held out, seed 42** — the queue is used on clients/pages the model has not seen, so rows from one client must not be in both sides.

In [2]:
tr, te = client_split(df)
print('train clients', df.client_id.iloc[tr].nunique(), 'test clients', df.client_id.iloc[te].nunique(), 'overlap', len(set(df.client_id.iloc[tr])&set(df.client_id.iloc[te])))

train clients 25 test clients 7 overlap 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same split, same metrics as the Week-4 baseline.

In [3]:
X = features(df); y = df.y.values
rows = {'baseline_rule': evaluate(y[te], baseline_score(df)[0].values[te])}
fitted = {}
for n,m in models().items():
    m.fit(X.iloc[tr], y[tr]); fitted[n]=m; rows[n]=evaluate(y[te], m.predict_proba(X.iloc[te])[:,1])
res = pd.DataFrame(rows).T; print(res)
import json; json.dump(rows, open(OUT/'model_metrics.json','w'), indent=1)

               roc_auc  avg_precision  p@50  p@200  base_rate       n
baseline_rule    0.463          0.470  0.36  0.325      0.511  6163.0
logreg           0.631          0.629  0.82  0.780      0.511  6163.0
random_forest    0.626          0.617  0.74  0.680      0.511  6163.0
hist_gbm         0.623          0.621  0.76  0.745      0.511  6163.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error analysis: permutation importance for the RF, and P@50 within low vs high visibility pages.

In [4]:
from sklearn.inspection import permutation_importance
rf = fitted['random_forest']
pi = permutation_importance(rf, X.iloc[te], y[te], scoring='roc_auc', n_repeats=3, random_state=SEED, n_jobs=-1)
print(pd.Series(pi.importances_mean, X.columns).sort_values(ascending=False).head(8).round(4))
p = rf.predict_proba(X.iloc[te])[:,1]; t = df.iloc[te]
for name, mask in [('<1k impr', t.impressions_90d<1000), ('>=1k impr', t.impressions_90d>=1000)]:
    print(name, evaluate(t.y.values[mask.values], p[mask.values]))

days_with_impressions    0.0466
avg_position             0.0229
content_age_days         0.0210
impressions_90d          0.0107
clicks_90d               0.0091
ctr                      0.0086
days_with_sessions       0.0064
sessions_90d             0.0043
dtype: float64
<1k impr {'roc_auc': 0.651, 'avg_precision': 0.643, 'p@50': 0.76, 'p@200': 0.75, 'base_rate': 0.52, 'n': 3777}
>=1k impr {'roc_auc': 0.581, 'avg_precision': 0.577, 'p@50': 0.62, 'p@200': 0.63, 'base_rate': 0.497, 'n': 2386}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.